<h1 style=\"text-align: center; font-size: 50px;\">  Text Generation with Neural Networks and Torch MLflow Integration</h1>

# Notebook Overview
- Start Execution
- User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

## Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  

logger.info("Notebook execution started.")

2025-12-01 18:29:46 - INFO - Notebook execution started.


## Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 61.1 ms, sys: 23.9 ms, total: 85 ms
Wall time: 2.02 s


In [4]:
# -----------------------------
# Standard library imports
# -----------------------------
import os                   # Operating system utilities (paths, env vars, etc.)
import sys                  # Python runtime environment manipulation
import time                 # Time-related utilities
import warnings             # Warning control and message handling
from datetime import datetime  # Date and time handling
from pathlib import Path     # Object-oriented filesystem paths

# -----------------------------
# Third-party imports
# -----------------------------
import numpy as np           # Numerical computations and arrays
import pandas as pd          # Data manipulation and analysis
import torch                 # PyTorch deep learning framework
import torch.nn.functional as F  # Functional API for neural network operations
from torch import nn          # Neural network layers and modules

import mlflow                 # ML lifecycle management and experiment tracking
from mlflow.models import ModelSignature  # MLflow model signature definition
from mlflow.types.schema import ColSpec, Schema  # MLflow schema utilities

# -----------------------------
# Local imports
# -----------------------------
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.mlflow import Logger

from src.utils import (
    load_config,
    download_from_s3_uri 
)

## User Constants

In [5]:
INITIAL_WORD = 'Love'
SIZE = 100

## Configure Settings

In [6]:
torch.manual_seed(0)

In [7]:
warnings.filterwarnings("ignore")

In [8]:
# ------------------------ Define global experiment and run names to be used throughout the notebook ------------------------
RUN_NAME = "RNN Text Generation"
MODEL_NAME = "dict_torch_rnn_model"
TORCH_MODEL = "dict_torch_rnn_model.pt"
EXPERIMENT_NAME = "Shakespeare Text Generation"
REGISTER_NAME = "Shakespeare_Model"

# ------------------------ Remote asset URIs ------------------------
S3_BASE = "s3://149536453923-hpaistudio-public-assets/AI-Blueprints/deep-learning/text-generation-with-rnn"
MODEL_URI = f"{S3_BASE}/{TORCH_MODEL}"
DATA_NAME = "shakespeare.txt"
DATA_URI = f"{S3_BASE}/{DATA_NAME}"

# ------------------------ Local paths ------------------------
ROOT = Path("..").resolve()
MODELS_PATH = ROOT / "models"
DATA_PATH = ROOT / "data" / DATA_NAME
MODEL_DECODER_PATH = ROOT / "models" / "decoder.pt"
MODEL_ENCODER_PATH = ROOT / "models" / "encoder.pt"
MODEL_STATE_PATH = ROOT / "models" / TORCH_MODEL
MODEL_PATH = ROOT / "models" / 'dict_torch_rnn_model.pt'
DEMO_FOLDER = ROOT / "demo"
CONFIG_PATH = ROOT / "configs" / "config.yaml"

# ------------------------ MLflow tracking ------------------------
# If your environment / platform uses a local MLflow path, set here.
# Compatible with Phoenix MLflow if mounted at this path.
MLFLOW_TRACKING_URI = "/phoenix/mlflow"

In [9]:
# Download model state dict and data (idempotent)
saved_model = download_from_s3_uri(MODEL_URI, MODEL_STATE_PATH.parent)
logger.info(f"Saved model state to: {saved_model}")

saved_data = download_from_s3_uri(DATA_URI, DATA_PATH.parent)
logger.info(f"Saved data to: {saved_data}")

2025-12-01 18:29:54 - INFO - Saved model state to: /home/jovyan/AI-Blueprints/deep-learning/text-generation-with-rnn/models/dict_torch_rnn_model.pt
2025-12-01 18:29:55 - INFO - Saved data to: /home/jovyan/AI-Blueprints/deep-learning/text-generation-with-rnn/data/shakespeare.txt


## Verify Assets

In [10]:
def log_asset_status(asset_path: str, asset_name: str, success_message: str, failure_message: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
        success_message (str): Message to log if asset exists.
        failure_message (str): Message to log if asset does not exist.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured. {success_message}")
    else:
        logger.error(f"{asset_name} is not properly configured. {failure_message}")
        
log_asset_status(
    asset_path=DATA_PATH,
    asset_name="Shakespeare text",
    success_message="",
    failure_message="Please run the 'run-workflow' notebook first and check if data folder was properly downloaded in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_DECODER_PATH ,
    asset_name="Decoder model",
    success_message="",
    failure_message="Please check if models folder was properly configured in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_ENCODER_PATH,
    asset_name="Encoder model",
    success_message="",
    failure_message="Please check if models folder was properly configured in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_STATE_PATH,
    asset_name="Rnn model",
    success_message="",
    failure_message="Please if models folder was properly downloaded in your project on AI Studio."
)
log_asset_status(
    asset_path=DEMO_FOLDER,
    asset_name="demo",
    success_message="",
    failure_message="Please check if demo folder was properly configured in your project on AI Studio."
)
log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="config",
    success_message="",
    failure_message="Please check if config file was properly configured in your project on AI Studio."
)

2025-12-01 18:29:55 - INFO - Shakespeare text is properly configured. 
2025-12-01 18:29:55 - INFO - Decoder model is properly configured. 
2025-12-01 18:29:55 - INFO - Encoder model is properly configured. 
2025-12-01 18:29:55 - INFO - Rnn model is properly configured. 
2025-12-01 18:29:55 - INFO - demo is properly configured. 
2025-12-01 18:29:55 - INFO - config is properly configured. 


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Creating the LSTM Model

In [12]:
# Get Text Data
with open(DATA_PATH,'r',encoding='utf8') as f:
    text = f.read()

In [13]:
all_characters = sorted(list(set(text))) # creates a set of unique characters found in the text

In [14]:
class CharModel(nn.Module):
    def __init__(self, decoder, encoder, all_chars, num_hidden=256, num_layers=4,drop_prob=0.5, use_gpu=False):
        """Initializes CharModel

        Args:
            decoder: Assigns a unique integer to each character in a dictionary format
            encoder : Reverses the decoder dictionary, providing a mapping from characters to their respective assigned integers.
            all_chars: Set of unique characters found in the text.
            num_hidden: Number of hidden layers. Defaults to 256.
            num_layers: Number of layers. Defaults to 4.
            drop_prob: Regularization technique to prevent overfitting. Defaults to 0.5.
            use_gpu: If the model uses GPU. Defaults to False.
        """
        try:
            super().__init__()
            self.drop_prob = drop_prob
            self.num_layers = num_layers
            self.num_hidden = num_hidden
            self.use_gpu = use_gpu
            
            self.all_chars = all_chars
            self.decoder = torch.load(decoder)
            self.encoder = torch.load(encoder)
            
            self.lstm = nn.LSTM(len(self.all_chars), num_hidden, num_layers, dropout=drop_prob, batch_first=True)
            self.dropout = nn.Dropout(drop_prob)
            self.fc_linear = nn.Linear(num_hidden, len(self.all_chars))
            logger.info("CharModel initialized successfully")
    
        except Exception as e:
            logger.error(f"Error initializing CharModel: {str(e)}")
      
    
    def forward(self, x, hidden):
        """Implementation of the CharModel logic, in which, the input passes through every step of the arquiteture

        Args:
            x: Input tensor with shape (batch size and senquency length) containing character indices.
            hidden: Tuple containing the inicial hidden states of the CharModel each with shape (batch size and senquency length).

        Returns:
            final_out: Output tensor representing the predicted logits for each character in the sequence.
            hidden: Tuple containing the final hidden states of the CharModel.
        """
        try:
            lstm_output, hidden = self.lstm(x, hidden)       
            drop_output = self.dropout(lstm_output)
            drop_output = drop_output.contiguous().view(-1, self.num_hidden)
            final_out = self.fc_linear(drop_output)
            
            return final_out, hidden
        
        except Exception as e:
            logger.error(f"Error implementing CharModel logic: {str(e)}")
    
    
    def hidden_state(self, batch_size):
        """
        Initializes and returns the initial hidden state for a recurrent neural network (e.g., LSTM).

        This method creates zero-filled tensors for the hidden state (h_0) and cell state (c_0), 
        supporting GPU execution if `self.use_gpu` is set to True.

        Args:
            batch_size: The number of sequences in the input batch, used to determine the tensor dimensions.

        Returns:
            Tuple: A tuple containing the hidden state and cell state tensors 
            with shape (num_layers, batch_size, num_hidden). Returns None if an exception occurs, and logs the error.
        """
        try:
            if self.use_gpu:
                hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden).to(device),
                        torch.zeros(self.num_layers,batch_size,self.num_hidden).to(device))
            else:
                hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden),
                        torch.zeros(self.num_layers,batch_size,self.num_hidden))
            
            return hidden
        except Exception as e:
            logger.error(f"Error Initializing and returning the initial hidden state: {str(e)}")

## Logging Model to MLflow

In [15]:
# Define input/output schema for the RNN text generation model
input_schema = Schema([
    ColSpec("string", "initial_word"),
    ColSpec("long", "size")
])

output_schema = Schema([
    ColSpec("string", "generated_text")
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)
logger.info("Model signature created successfully")

2025-12-01 18:29:55 - INFO - Model signature created successfully


In [16]:
mlflow.set_tracking_uri('/phoenix/mlflow')
mlflow.set_experiment(experiment_name= EXPERIMENT_NAME)

<Experiment: artifact_location='/phoenix/mlflow/305514499426307614', creation_time=1764608655438, experiment_id='305514499426307614', last_update_time=1764608655438, lifecycle_stage='active', name='Shakespeare Text Generation', tags={}>

In [17]:
model_state_dict = MODEL_PATH
register_name = REGISTER_NAME 

In [18]:
with mlflow.start_run(run_name = RUN_NAME) as run:
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Use new Logger with models-from-code approach
    Logger.log_model(
        signature=signature,
        model_state_dict_path=MODEL_PATH,
        decoder_path=MODEL_DECODER_PATH,
        encoder_path=MODEL_ENCODER_PATH,
        artifact_path=REGISTER_NAME,
        config_path=CONFIG_PATH,
        data_path=DATA_PATH,
        demo_folder=DEMO_FOLDER
    )
    
    mlflow.register_model(model_uri = f"runs:/{run.info.run_id}/{REGISTER_NAME}", name=register_name)

2025-12-01 18:29:56 - INFO - Run's Artifact URI: /phoenix/mlflow/305514499426307614/0ac8dc1e24b241e0bfde27a6bd1cb716/artifacts
Registered model 'Shakespeare_Model' already exists. Creating a new version of this model...
2025/12/01 18:30:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 0ac8dc1e24b241e0bfde27a6bd1cb716 has no artifacts at artifact path 'Shakespeare_Model', registering model based on models:/m-396dd75ba2ff4d5ba63c93c6bcf6d1ac instead
Created version '11' of model 'Shakespeare_Model'.


## Fetching the Latest Model Version from MLflow

In [19]:
client = mlflow.MlflowClient()
model_metadata = client.get_latest_versions(register_name, stages=["None"])
latest_model_version = model_metadata[0].version
latest_model_version

11

## Loading the Model and Running Inference

In [20]:
loaded = mlflow.pyfunc.load_model(model_uri=f"models:/{REGISTER_NAME}/{latest_model_version}")
# IMPORTANT: pyfunc expects a DataFrame input per our signature
in_df = pd.DataFrame({"initial_word": [INITIAL_WORD], "size": [SIZE]})
result_df = loaded.predict(in_df)
print(result_df)  # display the generated text

LoveENMN_jMtEDBMleMm,RDQjMRNMzRZNREDdfMMMM(_RN)ME5MEN|Mg_E5M-R5M_jMN_RNM5NRD45MRM5N,lDB)fMMMMg_RNM_RN_MN_


In [21]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-12-01 18:30:17 - INFO - ⏱️ Total execution time: 0m 31.77s
2025-12-01 18:30:17 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).